# Feature Selection

This tutorial demonstrates **stratification feature selection** for contact survey models
using `FeatureSelector`. The goal is to find the most parsimonious set of stratification
variables that explains the observed contact patterns.

**Method**: Leave-One-Out Cross-Validation Expected Log Predictive Density (LOO-CV ELPD),
computed via [`arviz.compare`](https://python.arviz.org/en/latest/api/generated/arviz.compare.html).

**Key rule**: All candidate models must stratify by a *subset* of the reference model's
variables. Every candidate is evaluated on the reference model's full observation grid so
that LOO scores are directly comparable.

In [ ]:
from cntmosaic.datasets import load_age_distribution, load_template_patterns
from cntmosaic.sim import (
    Stratification,
    Population,
    ParticipantSampler,
    MatrixSampler,
    ContactSampler,
)
from cntmosaic.dataloader import (
    ContactSurveyLoader,
    ParticipantData,
    ContactData,
    PopulationData,
    StratificationData,
)
from cntmosaic.models import GenMixFF, AgeMixFF
from cntmosaic.models.numpyro.priors import PSpline2D
from cntmosaic.analysis import FeatureSelector, ModelConfig
from cntmosaic.vis import plot_mosaic

from numpyro.infer.autoguide import AutoNormal
from numpyro.infer.initialization import init_to_value
from jax.random import PRNGKey

import altair as alt

## Generating Synthetic Data

In [ ]:
df_ref_pop = load_age_distribution("United_States", 80)

strats = [
    Stratification(
        name="sex",
        n_strata=2,
        ref_age_dist=df_ref_pop["P"].values,
        labels=["M", "F"],
        seed=0,
    ),
    Stratification(
        name="educ",
        n_strata=3,
        ref_age_dist=df_ref_pop["P"].values,
        labels=["Low", "Mid", "High"],
        seed=1,
    ),
]

In [ ]:
# Generate populations
pc = Population(strats)
df_pop = pc.df_P
df_pop_prop = pc.df_Q

In [ ]:
# Generate participants
pg = ParticipantSampler(pc, n_part=1500)
df_part = pg.sample(seed=0)

df_part.head(5)

In [ ]:
templates = load_template_patterns("United_States", smooth=True, max_age=80)

# Generate contact matrices
mg = MatrixSampler(templates)
cint_matrices = mg.generate_partial(pc, 10, seed=0)

In [ ]:
chart_list = []
for key, values in cint_matrices.items():
    if "M_" in key:
        chart = plot_mosaic(values, title=str(key), zlabel="Intensity")
        chart_list.append(chart)

alt.hconcat(*chart_list)

In [ ]:
chart_list = []
for key, values in cint_matrices.items():
    if "F" in key:
        chart = plot_mosaic(values, title=str(key), zlabel="Intensity")
        chart_list.append(chart)

alt.hconcat(*chart_list)

Here, we hack `cint_matrices` so that all education levels share the same contact pattern.
The feature selector should therefore choose the model that *does not* stratify by education
as the best model.

In [ ]:
# Collapse education-level contact matrices so educ adds no signal
cint_matrices["M_Low->All"] = cint_matrices["M_Mid->All"]
cint_matrices["M_High->All"] = cint_matrices["M_Mid->All"]

cint_matrices["F_Low->All"] = cint_matrices["F_Mid->All"]
cint_matrices["F_High->All"] = cint_matrices["F_Mid->All"]

In [ ]:
# Generate contacts
cg = ContactSampler(df_part, cint_matrices, "poisson", random_effects=True)
df_cnt = cg.sample(seed=0)

## Feature Selection

We compare three models with different stratification configurations:

| Name | `part_strat_vars` | `model_cls` | Description |
|------|-------------------|-------------|-------------|
| `sex_educ` | `["sex", "educ"]` | `GenMixFF` | Reference — most complex |
| `sex_only` | `["sex"]` | `GenMixFF` | Sex stratification only |
| `no_strat` | `[]` | `AgeMixFF` | Age-only baseline |

Because the contact matrices were constructed so that education adds no signal, the
feature selector should rank `sex_only` as the best model.

### Step 1: Build DataLoaders

Each model requires its own `ContactSurveyLoader` with the appropriate `strat_var_cols`.
`ParticipantData`, `PopulationData`, and `StratificationData` must all be consistent —
if a variable is dropped from `ParticipantData`, drop it from the others too.

In [ ]:
# Reference model: stratify by sex and education
loader_full = ContactSurveyLoader.from_containers(
    ParticipantData(df_part, id_col="id", age_col="age", strat_var_cols=["sex", "educ"]),
    ContactData(df_cnt, id_col="id", age_col="cnt_age"),
    PopulationData(df_pop, age_col="age", size_col="P", strat_var_cols=["sex", "educ"]),
    StratificationData(df_pop_prop, age_col="age", strat_var_cols=["sex", "educ"], prop_col="Q"),
)

# Candidate 1: sex stratification only
loader_sex = ContactSurveyLoader.from_containers(
    ParticipantData(df_part, id_col="id", age_col="age", strat_var_cols=["sex"]),
    ContactData(df_cnt, id_col="id", age_col="cnt_age"),
    PopulationData(df_pop, age_col="age", size_col="P", strat_var_cols=["sex"]),
    StratificationData(df_pop_prop, age_col="age", strat_var_cols=["sex"], prop_col="Q"),
)

# Candidate 2: no stratification
loader_simple = ContactSurveyLoader.from_containers(
    ParticipantData(df_part, id_col="id", age_col="age"),
    ContactData(df_cnt, id_col="id", age_col="cnt_age"),
    PopulationData(df_pop, age_col="age", size_col="P"),
)

### Step 2: Configure Models

`ModelConfig` bundles everything needed to instantiate and fit one model:

| Field | Type | Description |
|-------|------|-------------|
| `name` | `str` | Unique label in the comparison table |
| `model_cls` | `type` | `ContactModel` subclass (e.g. `GenMixFF`, `AgeMixFF`) |
| `dataloader` | `ContactSurveyLoader` | Pre-built loader for this configuration |
| `priors` | `dict` | `{variable: Prior2D}` — must include `"rate"` |
| `likelihood` | `str` | `"negbin"` (default) or `"poisson"` |
| `guide_factory` | `callable` | Optional per-model override of the guide factory |

The **guide factory** is a callable `guide_factory(model) -> guide` called once per model
after instantiation. It creates the variational guide, which typically needs `model.log_P`
for the `baseline` initialisation.

In [ ]:
def make_guide(model):
    """Standard AutoNormal guide with baseline initialisation."""
    return AutoNormal(
        model.model,
        init_loc_fn=init_to_value(values={"baseline": -float(model.log_P.mean())}),
    )

config_full = ModelConfig(
    name="sex_educ",
    model_cls=GenMixFF,
    dataloader=loader_full,
    priors={
        "rate": PSpline2D("global", M=15),
        "sex":  PSpline2D("partial", M=15),
        "educ": PSpline2D("partial", M=15),
    },
)

config_sex = ModelConfig(
    name="sex_only",
    model_cls=GenMixFF,
    dataloader=loader_sex,
    priors={
        "rate": PSpline2D("global", M=15),
        "sex":  PSpline2D("partial", M=15),
    },
)

config_simple = ModelConfig(
    name="no_strat",
    model_cls=AgeMixFF,
    dataloader=loader_simple,
    priors={"rate": PSpline2D("global", M=15)},
)

### Step 3: Run `FeatureSelector`

`FeatureSelector` takes a **reference config** (most complex model) and a list of
**candidate configs** (simpler subsets). It:

1. Validates that each candidate's stratification variables are a subset of the reference's
2. Fits all models via SVI using independent splits of the provided `PRNGKey`
3. Evaluates every candidate on the **reference model's observation grid**
   (not its own reduced grid) so LOO scores are on the same scale
4. Returns a `FeatureSelectionResult` with the ranked comparison table

Key constructor parameters:

| Parameter | Default | Description |
|-----------|---------|-------------|
| `guide_factory` | — | Shared guide factory; overridable per `ModelConfig` |
| `num_steps` | `5_000` | SVI optimisation steps applied to all models |
| `peak_lr` | `0.01` | Peak learning rate for the cosine-annealing schedule |
| `num_samples` | `1_000` | Posterior samples drawn per model for LOO computation |

In [ ]:
selector = FeatureSelector(
    reference_config=config_full,
    candidate_configs=[config_sex, config_simple],
    guide_factory=make_guide,
    num_steps=5_000,
)

result = selector.run(PRNGKey(0))

## Results

`FeatureSelector.run()` returns a `FeatureSelectionResult` with the following attributes:

| Attribute | Type | Description |
|-----------|------|-------------|
| `result.comparison` | `pd.DataFrame` | `arviz.compare()` table, sorted by ELPD-LOO (best first) |
| `result.models` | `dict[str, ContactModel]` | Fitted model instances keyed by name |
| `result.idatas` | `dict[str, InferenceData]` | ArviZ `InferenceData` objects keyed by name |
| `result.best_model` | `ContactModel` | Convenience property: model with highest ELPD-LOO |

The comparison table uses ArviZ conventions:
- **`elpd_loo`**: expected log pointwise predictive density (higher is better)
- **`p_loo`**: effective number of parameters
- **`elpd_diff`**: difference from the best model's ELPD (0 for the best model)
- **`weight`**: stacking weight

In [ ]:
result.comparison

In [ ]:
print("Best model:", result.comparison.index[0])

# Retrieve the fitted model for downstream analysis (e.g. ModelSummariser)
best = result.best_model

# Retrieve InferenceData for an individual model
idata_sex = result.idatas["sex_only"]